## XGBoost

Grid Search is used to optimize the boosting parameters and evaluate the model's performance.

In [ ]:
import pandas as pd

from daa.config import OUTPUTS_DIR
from daa.data import load_train
from daa.experiment import run_experiment
from daa.splits import get_folds



### Evaluate XGBoost using the selected features and fit a final model on the complete training dataset.

In [ ]:
X, y = load_train()
folds = get_folds(y)
print(f"real data: {X.shape[0]:,} rows x {X.shape[1]:,} columns")
print(f"{len(folds)} folds, {y.nunique()} classes")


# Run the XGBoost experiment using the selected features.
Note that the model may fail if any feature name contains unsupported characters.

In [ ]:
XGBoost = run_experiment(
    "xgboost",
    name="XGBoost",
    X=X,
    y=y,
    folds=folds,
    feature_set="selected",
    fit_full=True,
)
pd.Series(XGBoost.metrics, name=XGBoost.name)



## Feature Check
Identify column names containing potentially problematic characters before fitting the model.



In [ ]:
bad_columns = [
    col for col in X.columns
    if any(char in str(col) for char in ["[", "]", "<"])
]

print("Colunas problemáticas:", bad_columns)

The result [] means that no column X name contains [, ] and <.

This is important because it means the error from XGBoost does not appear to lie in the original column names of X.

It is possible that the problematic names are generated during preprocessing—for example, when new features are created from categorical variables (via One-Hot Encoding).

# Results
Retrieve the XGBoost experiment results from the leaderboard, including the number of features and cross-validation performance.

In [ ]:
leaderboard = pd.read_csv(OUTPUTS_DIR / "leaderboard.csv")
score_columns = ["run_id", "name", "n_features", "macro_f1_mean", "macro_f1_std"]
leaderboard[leaderboard["name"].str.contains("xgboost", case=False)][
    score_columns
]